# Log-Linear Models for Contingency Tables

## Overview
Chi-square tests handle 2-way contingency tables. **Log-linear models** extend this to 3-way and higher-order tables, modelling the expected cell frequencies as a log-linear function of main effects and interactions — analogous to ANOVA for frequency data.

| Table type | Approach |
|---|---|
| 2-way | Chi-square or log-linear (equivalent) |
| 3-way | Log-linear model required |
| Any | Log-linear handles unbalanced cells, structural zeros, ordinal variables |

**Quinn & Keough (2002) ch. 14.**

---

In [ ]:
library(tidyverse); library(MASS); library(vcd)
# 3-way contingency table: seabird breeding success
# Variables: Species (3), Site type (2: island/mainland), Year (2: before/after)
set.seed(9)
n_cells <- 12   # 3 × 2 × 2
# Simulate count table with species × site interaction
ftable_data <- array(
  c(45,12, 30,18, 22,35,   # before: island vs mainland
    50,10, 28,22, 18,40),  # after
  dim = c(3, 2, 2),
  dimnames = list(
    Species  = c("Tern","Puffin","Gannet"),
    SiteType = c("Island","Mainland"),
    Year     = c("Before","After")
  )
)
cat("3-way contingency table:\n")
print(ftable(ftable_data))

In [ ]:
# Convert to data frame for lm-style fitting
df <- as.data.frame(as.table(ftable_data))
names(df)[4] <- "Count"
cat("Data frame for log-linear modelling:\n"); print(head(df,8))

# Fit log-linear models of increasing complexity
# Mutual independence: all variables independent
m_indep   <- glm(Count ~ Species + SiteType + Year,
                 data=df, family=poisson)
# All two-way interactions (no 3-way)
m_2way    <- glm(Count ~ (Species + SiteType + Year)^2,
                 data=df, family=poisson)
# Saturated (perfect fit)
m_sat     <- glm(Count ~ Species * SiteType * Year,
                 data=df, family=poisson)

cat("\nModel comparison (AIC):\n")
AIC_tab <- AIC(m_indep, m_2way, m_sat)
print(AIC_tab)
cat("\nDeviance goodness-of-fit:\n")
cat("  Independence model deviance:", round(deviance(m_indep),2),
    " df:", df.residual(m_indep), "\n")
cat("  2-way interaction deviance: ", round(deviance(m_2way),2),
    " df:", df.residual(m_2way), "\n")

In [ ]:
# Test specific interactions
cat("LRT: does 3-way interaction improve fit?\n")
print(anova(m_2way, m_sat, test="LRT"))

cat("\nLRT: does Species:SiteType interaction explain data?\n")
m_no_sp_site <- glm(Count ~ Species + SiteType + Year + Species:Year + SiteType:Year,
                    data=df, family=poisson)
print(anova(m_no_sp_site, m_2way, test="LRT"))

# Interpret: fitted values are expected cell frequencies under the model
cat("\nFitted vs observed frequencies under 2-way model:\n")
df$fitted <- round(fitted(m_2way), 1)
df$residual_std <- rstandard(m_2way)
print(df[order(abs(df$residual_std), decreasing=TRUE)[1:6],])
cat("Large standardised residuals indicate poorly fitted cells\n")

---
## Common Pitfalls

**1. Using chi-square for 3-way tables**
A simple chi-square test cannot decompose the association structure of a 3-way table — it can only test overall independence. Log-linear models are required to identify which two-way and three-way interactions are present. Applying chi-square separately to two-way margins of a 3-way table ignores the third variable entirely.

**2. Not checking expected cell frequencies before fitting**
Log-linear models (like chi-square) require adequate expected cell frequencies — commonly ≥ 5 per cell. Sparse tables with many small or zero cells can cause fitted models to be unreliable. Collapse categories or use exact methods for sparse data.

**3. Interpreting a significant 3-way interaction without examining lower-order effects**
A significant 3-way interaction (A:B:C) means the 2-way A:B association differs across levels of C. Always test and interpret lower-order interactions before interpreting higher-order ones, just as in factorial ANOVA.

**4. Treating log-linear models as regression**
In log-linear models, all variables are treated as response variables whose joint frequency distribution is being modelled — there is no designated response variable (unlike logistic regression). If you have a clear response (binary outcome), logistic regression is more appropriate.

**5. Forgetting that the saturated model always fits perfectly**
The saturated log-linear model (all interactions included) reproduces observed frequencies exactly with zero deviance. It is not a useful model for inference — always compare simpler models to the saturated model using LRT or AIC.

**6. Using structural zeros vs sampling zeros interchangeably**
A structural zero (a combination that is logically impossible, e.g., male pregnancy) must be handled differently from a sampling zero (a possible combination not observed by chance). Structural zeros require special treatment in the model; ignoring them distorts the analysis.


---
*r_methods_library - Samantha McGarrigle*